# MA12003 Year 1 S2 Programming — Working Notebook

To assess your coursework submission, I will run your **working notebook**.

**Before making any changes to this notebook, rename it to `working_notebook_USERNAME.ipynb`** where `USERNAME` is your university username (e.g. `working_notebook_gh3145.ipynb`).

- Do **not** add or remove any cells in this notebook.
- Write your code in the **[Demonstrate]** cells and your written answers in the **[Justify]** cells.
- All imports must go in the **Imports** section below.

See `coursework_specification.ipynb` for submission instructions, assessment criteria, and task descriptions.

---
#### Enter your full name and Bath user ID (e.g. gh3145) in the cell below:

Name: Nik Markov

User ID: nm2263

**GenAI statement:** You must include a short statement (max 250 words) in the cell
below indicating what tools you used and how, or that you have not used genAI.

## **Imports**

All imports must appear in the cell below. **Importing packages outside this section will be penalised.**

In [2]:
from abc import ABC, abstractmethod
import math
import numpy as np
from numpy.linalg import det
from sympy import Matrix
from cipher_helpers import run_cipher_tests
from cipher_data import SECRET_MESSAGES as messages, CIPHER_ALPHABETS as alphabets

---
# Part A: Foundational Cipher Implementation &nbsp;&nbsp; [8 marks]

## Task A1: Caesar Cipher $C_c$ &nbsp;&nbsp; [2 marks]

In [5]:
class CipherBase(ABC):
    """Parent class for all ciphers, handling char maps, and case preservation."""
    
    DEFAULT_ALPHABET = "abcdefghijklmnopqrstuvwxyz"
    
    def __init__(self, alphabet=None):
        alphabet = CipherBase.DEFAULT_ALPHABET if alphabet is None else alphabet
        if not isinstance(alphabet, str):
            raise TypeError("alphabet must be string")
        
        alphabet_set = set(alphabet)
        
        if not len(alphabet_set) == len(alphabet):
            raise ValueError("Cipher alphabet must contain entirely distinct characters.")
        
        self.index_map = {ch: i for i, ch in enumerate(alphabet)}
        self.alphabet = alphabet
        self.alphabet_set = alphabet_set
        self.n = len(alphabet)

    def _transform_char(self, char, shift_func):
        """Preserves capitalisation and ignores non-alphabet characters."""
        
        lower = char.lower()
        idx = self.index_map.get(lower)
        if idx is None:
            return char

        new_char = self.alphabet[shift_func(idx) % self.n]
        return new_char.upper() if char.isupper() else new_char
       
    def encode(self, text):
        """Encodes the given text."""
        return "".join(self._transform_char(c, self.encode_idx) for c in text)
    
    def decode(self, text):
        """Decodes the given text."""
        return "".join(self._transform_char(c, self.decode_idx) for c in text)

    @abstractmethod
    def encode_idx(self, idx):
        """Child classes MUST define the mathematical encoding formula."""
        pass
    
    @abstractmethod
    def decode_idx(self, idx):
        """Child classes MUST define the mathematical decoding formula."""
        pass

class CaesarCipher(CipherBase):
    def __init__(self, shift, alphabet = None):
        super().__init__(alphabet)
        self.shift = shift

    def encode_idx(self, idx):
        return idx + self.shift

    def decode_idx(self, idx):
        return idx - self.shift

    def __str__(self):
        return f"CaesarCipher(shift={self.shift}, alphabet='{self.alphabet}')"

ciphers_a1 = [CaesarCipher(3), CaesarCipher(-5)]
run_cipher_tests(ciphers_a1, messages)


Cipher: CaesarCipher(shift=3, alphabet='abcdefghijklmnopqrstuvwxyz')
[bond] Original: Agent 007: "Shaken, not stirred!"
[bond] Encoded : Djhqw 007: "Vkdnhq, qrw vwluuhg!"
[bond] Decoded : Agent 007: "Shaken, not stirred!"
--------------------------------------------------
[holmes] Original: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
[holmes] Encoded : [Dw 221E Ednhu Vwuhhw] Krophv (wr Zdwvrq): "Hohphqwdub, pb ghdu Zdwvrq!"
[holmes] Decoded : [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
--------------------------------------------------
[joker] Original: Why so serious?! Let's put a smile on that face.
[joker] Encoded : Zkb vr vhulrxv?! Ohw'v sxw d vploh rq wkdw idfh.
[joker] Decoded : Why so serious?! Let's put a smile on that face.
--------------------------------------------------
[turing] Original: In 1950, Turing asked: "Can machines think?"
[turing] Encoded : Lq 1950, Wxulqj dvnhg: "Fdq pdfklqhv wklqn?"
[turing] Decoded : 

**[Justify]**

My initial design choice was to structure the solution around an abstract base class `Cipher` and a subclass `CaesarCipher` to keep my code well-structured and modular: `Cipher` handles the behaviour shared by every character-based cipher, while `CaesarCipher` only supplies the specific index mapping for shifting letters. The result is less duplicated code for the later defined classes that follow the same text processing logic, easier testing since case handling, punctuation, and other common string behaviours pass through one shared class, and adding future behaviour later becomes much cleaner and easier to implement.

My second design choice was to use abstract methods for `encode_idx` and `decode_idx` because they enforce every cipher to define both encoding and decoding behaviour. This is useful for correctness: a subclass cannot be instantiated unless it supplies its required mappings. It is also useful for reuse: the same `encode` and `decode` methods in the parent class can operate on any valid child cipher without changing the `run_cipher_tests` test code.

Another design choice was to represent the alphabet as an ordered string and a set. The ordered string is needed for index lookup and reconstruction of characters, while the set provides fast checks when deciding whether a character should be transformed or left unchanged. Using both keeps the logic clean and the code more modular: one structure for transformation, one for validation.

A further choice that I implemented was to build encoded and decoded strings using `str.join(...)` with a generator expression. I considered using string concatenation but, since strings are immutable, repeated concatenation in a loop would have led to a bunch of temporary strings which is less efficient. Also, the generator expression keeps the transformation step readable: each character is mapped and then joined into the final string.

Finally, in the `__init__` method I chose to raise a `ValueError` if the alphabet contains repeated characters; that prevents ambiguous indexing and ensures the cipher is well-defined before any encoding begins.

**How to decode a Caesar cipher**

For a Caesar cipher $C_c$ on an alphabet of length $n$, encoding a character at index $i$ gives
$$
E(i) = (i + c) \bmod n.
$$

To decode, we reverse the shift by subtracting the same amount:
$$
D(j) = (j - c) \bmod n.
$$

So decoding is simply the inverse modular shift of the encoding rule. This works because modular arithmetic cycles values back into the valid index range, so every encoded character maps back to exactly one original character.

For example, using $C_3$ on the default alphabet:
- `a` has index $0$, so encoding gives $(0 + 3) \bmod 26 = 3$, which is `d`.
- To decode `d`, we compute $(3 - 3) \bmod 26 = 0$, which returns `a`.
- The cyclic case is `c`, which has index $2$. Decoding under $C_3$ gives $(2 - 3) \bmod 26 = -1 \equiv 25 \pmod{26}$, which is `z`.

## Task A2: Affine Cipher $A_{a,b}$ &nbsp;&nbsp; [2 marks]

In [2]:
class AffineCipher(Cipher):
    def __init__(self, multiplier, offset, alphabet="abcdefghijklmnopqrstuvwxyz"):
        super().__init__(alphabet)

        if math.gcd(multiplier, self.n) != 1:
            raise ValueError(f"Multiplier {multiplier} is not coprime with alphabet length {self.n}.")
        
        self.a = multiplier
        self.b = offset
        self._inv_a = pow(self.a, -1, self.n)

    def encode_idx(self, idx):
        return self.a * idx + self.b

    def decode_idx(self, idx):
        return (self._inv_a * (idx - self.b)) % self.n

    def __str__(self):
        return f"AffineCipher(multiplier={self.a}, offset={self.b}, alphabet='{self.alphabet}')"

ciphers_a2 = [AffineCipher(7, 3), AffineCipher(5, 8)]
run_cipher_tests(ciphers_a2, messages)

NameError: name 'Cipher' is not defined

**[Justify]**

I kept the affine implementation class-structured rather than split into separate Python files: the parent `Cipher` class handles text traversal, case preservation, and non-alphabet characters, while `AffineCipher` only defines index transformations. This reduces duplication and makes each class easier to test.

I cache the modular inverse once in `__init__` as `_inv_a = pow(self.a, -1, self.n)`. This is efficient because decoding uses the same inverse repeatedly, so precomputing avoids unnecessary recalculation.

A validity check is required: `math.gcd(self.a, self.n) == 1`. This ensures `a` is invertible modulo `n`; without this, decoding is not uniquely possible. The offset `b` does not affect invertibility.

**How decoding works**

For affine cipher $A_{a,b}$:
$$
E(x) = ax + b \pmod n,
$$
so decoding is
$$
D(y) = a^{-1}(y - b) \pmod n.
$$

This inverse exists exactly when $\gcd(a,n)=1$.

**Concrete examples**

For $A_{7,3}$ over 26 letters, $7^{-1} \equiv 15 \pmod{26}$. If `a` has index $0$, encoding gives $(7\cdot 0 + 3) \bmod 26 = 3$ (`d`), and decoding gives $15(3 - 3) \bmod 26 = 0$ (`a`).

For $A_{5,8}$, $5^{-1} \equiv 21 \pmod{26}$. If `a` has index $0$, encoding gives $(5\cdot 0 + 8) \bmod 26 = 8$ (`i`), and decoding gives $21(8 - 8) \bmod 26 = 0$ (`a`).

If $a=2$ with $n=26$, then $\gcd(2,26)=2$, so the mapping is not bijective and unique decoding fails.

**Error handling:** `AffineCipher.__init__` raises `ValueError` when `math.gcd(multiplier, alphabet_length) != 1`, preventing invalid affine ciphers from being created.

## Task A3: Substitution Cipher $S_p$ &nbsp;&nbsp; [2 marks]

In [5]:
class SubstitutionCipher(Cipher):
    """Implementation of the general Substitution cipher S_sigma."""
    
    def __init__(self, permuted_alphabet, alphabet="abcdefghijklmnopqrstuvwxyz"):
        super().__init__(alphabet)
        
        if len(permuted_alphabet) != self.n or set(permuted_alphabet) != self.alphabet_set:
            raise ValueError("Permuted alphabet must contain the exact same characters as the base alphabet.")
            
        self.permuted_alphabet = permuted_alphabet
        self.sigma = [self.alphabet.index(c) for c in self.permuted_alphabet]
        
        self.inv_sigma = [0] * self.n
        for original_idx, permuted_idx in enumerate(self.sigma):
            self.inv_sigma[permuted_idx] = original_idx

    @staticmethod
    def reverse_permutation(alphabet):
        """Generate reverse permutation of alphabet."""
        return alphabet[::-1]
    
    @staticmethod
    def split_permutation(alphabet):
        """Generate split permutation of alphabet (evens first, then odds reversed)."""
        return alphabet[::2] + alphabet[1::2][::-1]

    def encode_idx(self, idx):
        return self.sigma[idx]

    def decode_idx(self, idx):
        return self.inv_sigma[idx]

    def __str__(self):
        return f"SubstitutionCipher(perm='{self.permuted_alphabet}', alphabet='{self.alphabet}')"

alphabet = "abcdefghijklmnopqrstuvwxyz"

sigma_rev_alphabet = SubstitutionCipher.reverse_permutation(alphabet)
sigma_split_alphabet = SubstitutionCipher.split_permutation(alphabet)

ciphers_a3 = [
    SubstitutionCipher(sigma_rev_alphabet),
    SubstitutionCipher(sigma_split_alphabet)
]

run_cipher_tests(ciphers_a3, messages)


Cipher: SubstitutionCipher(perm='zyxwvutsrqponmlkjihgfedcba', alphabet='abcdefghijklmnopqrstuvwxyz')
[bond] Original: Agent 007: "Shaken, not stirred!"
[bond] Encoded : Ztvmg 007: "Hszpvm, mlg hgriivw!"
[bond] Decoded : Agent 007: "Shaken, not stirred!"
--------------------------------------------------
[holmes] Original: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
[holmes] Encoded : [Zg 221Y Yzpvi Hgivvg] Slonvh (gl Dzghlm): "Vovnvmgzib, nb wvzi Dzghlm!"
[holmes] Decoded : [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
--------------------------------------------------
[joker] Original: Why so serious?! Let's put a smile on that face.
[joker] Encoded : Dsb hl hvirlfh?! Ovg'h kfg z hnrov lm gszg uzxv.
[joker] Decoded : Why so serious?! Let's put a smile on that face.
--------------------------------------------------
[turing] Original: In 1950, Turing asked: "Can machines think?"
[turing] Encoded : Rm 1950, Gfirmt zhpvw: "Xzm nzxs

**[Justify]**

I kept the same overall design as in A1 and A2: the parent `Cipher` class provides the shared text-handling logic, and `SubstitutionCipher` focuses only on the permutation itself. That is the main design benefit here, because substitution ciphers differ mathematically from Caesar and affine ciphers, but they still need the same surrounding machinery for case preservation and skipping characters outside the alphabet.

I store the substitution as two precomputed lookup structures: the forward permutation `self.sigma` and the inverse permutation `self.inv_sigma`. This is a strong design choice because encoding and decoding both become $O(1)$ per character after initialisation. It also makes the direction of the mapping explicit, which is much clearer than trying to reconstruct the inverse on the fly every time text is decoded.

I used `@staticmethod` for `reverse_permutation` and `split_permutation` because these helpers do not depend on instance state (`self`) or class state (`cls`): they are pure functions from an input alphabet string to a permuted alphabet string. Marking them as static methods makes that intent explicit, improves readability, and keeps the API clean (`SubstitutionCipher.reverse_permutation(alphabet)`) without requiring a temporary cipher object just to generate a permutation.

The bijection check uses set comparison: `set(permuted_alphabet) == self.alphabet_set`, together with a length check. This is important because a substitution cipher must be a one-to-one mapping. If a letter were missing or duplicated, the cipher would no longer be reversible. Using sets makes the validation concise and fast, and it directly matches the mathematical definition that $p$ must be a bijection.

**How decoding works**

A substitution cipher $S_p$ encodes each index $i$ by sending it to $p(i)$. To decode, we apply the inverse bijection $p^{-1}$:
$$
D(p(i)) = i.
$$

So decoding is not a new cipher rule; it is simply the inverse permutation. In code, that is exactly what `self.inv_sigma` stores.

**Concrete examples**

For $S_{\texttt{reverse}}$ on the default alphabet, the mapping reverses the order of the letters. That means `a` goes to `z`, `m` goes to `n`, and `z` goes to `a`. Because reversing twice returns the original order, decoding `z` gives `a` immediately, and the cipher is its own inverse.
For $S_{\texttt{split}}$, the alphabet is rearranged by taking letters in alternating positions and then reversing the odd positions. The exact output order is a valid permutation of the default alphabet, so decoding works the same way: whatever letter appears at position $p(i)$ during encoding is sent back to position $i$ during decoding by the inverse table.

This implementation therefore satisfies the core requirement of a substitution cipher: every valid output symbol has exactly one preimage, so encryption and decryption are perfectly reversible.

**Error handling clarity:** `SubstitutionCipher.__init__` raises a `ValueError` when the permuted alphabet is not a true bijection of the base alphabet (wrong length, missing characters, or duplicates).

## Task A4: Custom Alphabets &nbsp;&nbsp; [2 marks]

In [ ]:
qwerty_rev = alphabets["qwerty"][::-1]
ambidextrous_split = alphabets["ambidextrous"][::2] + alphabets["ambidextrous"][1::2][::-1]

ciphers_a4 = [
    CaesarCipher(shift=3, alphabet=alphabets["vowels"]),
    AffineCipher(multiplier=2, offset=5, alphabet=alphabets["consonants"]),
    SubstitutionCipher(permuted_alphabet=qwerty_rev, alphabet=alphabets["qwerty"]),
    SubstitutionCipher(permuted_alphabet=ambidextrous_split, alphabet=alphabets["ambidextrous"])
]

run_cipher_tests(ciphers_a4, messages)


Cipher: CaesarCipher(shift=3, alphabet='aeiou')
[bond] Original: Agent 007: "Shaken, not stirred!"
[bond] Encoded : Ogunt 007: "Shokun, net starrud!"
[bond] Decoded : Agent 007: "Shaken, not stirred!"
--------------------------------------------------
[holmes] Original: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
[holmes] Encoded : [Ot 221B Bokur Struut] Helmus (te Wotsen): "Ulumuntory, my duor Wotsen!"
[holmes] Decoded : [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
--------------------------------------------------
[joker] Original: Why so serious?! Let's put a smile on that face.
[joker] Encoded : Why se suraeis?! Lut's pit o smalu en thot focu.
[joker] Decoded : Why so serious?! Let's put a smile on that face.
--------------------------------------------------
[turing] Original: In 1950, Turing asked: "Can machines think?"
[turing] Encoded : An 1950, Tirang oskud: "Con mochanus thank?"
[turing] Decoded : In 1950, Turing asked

**[Justify]**

This task demonstrates that the cipher implementation is not hard-coded to the default alphabet. I designed the classes so that the alphabet is always an input parameter, and all index arithmetic depends on `self.n`, the actual length of that chosen alphabet. That means the same code can work for vowels, consonants, qwerty, or any other valid cipher alphabet without rewriting the cipher logic.

The use of a set remains important here as well. Once the alphabet changes, membership checks still need to be fast and reliable, and the set representation lets the code decide quickly whether a character belongs to the cipher alphabet. This matters because the messages contain punctuation, spaces, and digits that should pass through unchanged.

The main mathematical benefit of parameterising the alphabet is that all modular arithmetic automatically adapts to the correct size. For example, the vowels alphabet has length $5$, the consonants alphabet has length $21$, and qwerty has length $26$. The same formulas from A1--A3 still work because the code always reduces indices modulo the current alphabet length.

**Why these examples work well**

I chose several different alphabets to show genuine generality. `vowels` is short and makes the modular wrap-around easy to see. `consonants` shows that the cipher can work on a non-contiguous subset of letters. `qwerty` and `ambidextrous` demonstrate that the implementation does not care about the semantic meaning of the alphabet; it only needs a valid ordered sequence of distinct characters.

This is important because it proves the implementation is reusable rather than specialised. A correct design should not only work on `abcdefghijklmnopqrstuvwxyz`, but on any valid cipher alphabet supplied by the user, and that is exactly what the constructor-based alphabet parameter achieves.

**Error handling clarity:** invalid alphabet choices are rejected via explicit `ValueError` checks in constructors (for repeated characters, invalid substitution permutations, or non-coprime affine multipliers), which keeps custom-alphabet usage safe and unambiguous.

---
# Part B: Cipher Manipulation and Composition &nbsp;&nbsp; [8 marks]

### Task B1: Operations on Caesar Ciphers &nbsp;&nbsp; [2 marks]

In [ ]:
class CompositeCipher:
    """Handles R2 o R1 composition using the Composite Design Pattern."""
    
    def __init__(self, r2, r1):
        self.r2 = r2
        self.r1 = r1

    def encode(self, text):
        return self.r2.encode(self.r1.encode(text))

    def decode(self, text):
        return self.r1.decode(self.r2.decode(text))

    def __matmul__(self, other):
        return CompositeCipher(self, other)

    def __str__(self):
        return f"({self.r2} \u2218 {self.r1})"

# Inject composition into Cipher without redefining existing classes
Cipher.__matmul__ = lambda self, other: CompositeCipher(self, other)

# Add Task B1 methods to existing CaesarCipher class
def caesar_add(self, other):
    if not isinstance(other, CaesarCipher):
        raise TypeError("Can only add another CaesarCipher.")
    if self.alphabet != other.alphabet:
        raise ValueError("Forbidden: Addition requires identical cipher alphabets.")
    return CaesarCipher(self.shift + other.shift, self.alphabet)

def caesar_mul(self, n):
    if not isinstance(n, int):
        raise TypeError("Scalar multiplication requires an integer.")
    return CaesarCipher(self.shift * n, self.alphabet)

def caesar_rmul(self, n):
    return self.__mul__(n)

def caesar_neg(self):
    return CaesarCipher(-self.shift, self.alphabet)

def caesar_pow(self, n):
    if not isinstance(n, int) or n < 0:
        raise ValueError("Power must be a non-negative integer.")
    return CaesarCipher(self.shift * n, self.alphabet)

def caesar_str(self):
    return f"C_({self.shift})"

CaesarCipher.__add__ = caesar_add
CaesarCipher.__mul__ = caesar_mul
CaesarCipher.__rmul__ = caesar_rmul
CaesarCipher.__neg__ = caesar_neg
CaesarCipher.__pow__ = caesar_pow
CaesarCipher.__str__ = caesar_str

test_1 = CaesarCipher(3, alphabets["vowels"]) @ CaesarCipher(-5, alphabets["vowels"])
test_2 = CaesarCipher(3, alphabets["ambidextrous"]) ** 4
test_3 = 5 * CaesarCipher(2)
test_4 = -CaesarCipher(2, alphabets["vowels"])
test_5 = CaesarCipher(3, alphabets["qwerty"]) + CaesarCipher(4, alphabets["qwerty"])
test_6 = CaesarCipher(3, alphabets["vowels"]) @ CaesarCipher(5, alphabets["consonants"])

ciphers_b1 = [test_1, test_2, test_3, test_4, test_5, test_6]
run_cipher_tests(ciphers_b1, messages)


Cipher: (C_(3) ∘ C_(-5))
[bond] Original: Agent 007: "Shaken, not stirred!"
[bond] Encoded : Ogunt 007: "Shokun, net starrud!"
[bond] Decoded : Agent 007: "Shaken, not stirred!"
--------------------------------------------------
[holmes] Original: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
[holmes] Encoded : [Ot 221B Bokur Struut] Helmus (te Wotsen): "Ulumuntory, my duor Wotsen!"
[holmes] Decoded : [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
--------------------------------------------------
[joker] Original: Why so serious?! Let's put a smile on that face.
[joker] Encoded : Why se suraeis?! Lut's pit o smalu en thot focu.
[joker] Decoded : Why so serious?! Let's put a smile on that face.
--------------------------------------------------
[turing] Original: In 1950, Turing asked: "Can machines think?"
[turing] Encoded : An 1950, Tirang oskud: "Con mochanus thank?"
[turing] Decoded : In 1950, Turing asked: "Can machines think?"

: 

**[Justify]**

The core design choice in B1 was to model Caesar operations through Python operator overloading so the code matches the task notation directly (`+`, `*`, unary `-`, `**`, `@`). This improves readability and makes the implementation easier to test, because each algebraic operation is a separate method with one clear responsibility.

A second design choice was to extend the existing classes dynamically rather than rewrite them. Assignments such as `CaesarCipher.__add__ = caesar_add` attach the new behaviour to the class after the base cipher implementation is already complete. The benefit is that I can add B1 functionality incrementally without duplicating or restructuring earlier A1 code, reducing the risk of breaking earlier working methods.

I used the same dynamic approach for composition (`Cipher.__matmul__ = ...`) so every cipher subclass can use `@` consistently. This keeps the interface uniform across tasks: once composition is defined at the base level, Caesar, affine, and substitution ciphers all share the same composition syntax.

`CompositeCipher` is the wrapper class, and each use of `@` creates a `CompositeCipher` instance representing a composition $R_2 \circ R_1$. Its benefit is that individual cipher classes only need to know how to encode/decode themselves, while `CompositeCipher` handles chaining order. In particular, it guarantees correct inverse order during decoding (decode with the outer cipher first, then the inner one), which is easy to get wrong if composition is implemented ad hoc in each class.

For constraints, addition/subtraction enforce same-alphabet compatibility, while composition intentionally allows different alphabets as required by the specification. This logic is embedded directly in the operator methods, so invalid arithmetic is rejected at the operation point rather than failing later in tests.

**Error handling clarity:** B1 methods raise explicit exceptions: `TypeError` for invalid operand/scalar types and `ValueError` for forbidden same-alphabet violations or invalid powers.

## Task B2: Operations on Affine Ciphers &nbsp;&nbsp; [2 marks]

In [8]:
# Add Task B2 methods to the existing AffineCipher class

def affine_add(self, other):
    if not isinstance(other, AffineCipher):
        raise TypeError("Can only add another AffineCipher.")
    if self.alphabet != other.alphabet:
        raise ValueError("Forbidden: Addition requires identical cipher alphabets.")
    if self.a != other.a:
        raise ValueError("Forbidden: Addition requires identical multipliers (a).")
    return AffineCipher(self.a, self.b + other.b, self.alphabet)

def affine_mul(self, n):
    if not isinstance(n, int):
        raise TypeError("Scalar multiplication requires an integer.")
    return AffineCipher(self.a, self.b * n, self.alphabet)

def affine_rmul(self, n):
    return self.__mul__(n)

def affine_neg(self):
    return AffineCipher(self.a, -self.b, self.alphabet)

def affine_pow(self, n):
    if not isinstance(n, int) or n < 0:
        raise ValueError("Power must be a non-negative integer.")
    if n == 0:
        return AffineCipher(1, 0, self.alphabet)

    new_a, new_b = self.a, self.b
    for _ in range(n - 1):
        new_a = (self.a * new_a) % self.n
        new_b = (self.a * new_b + self.b) % self.n

    return AffineCipher(new_a, new_b, self.alphabet)

def affine_str(self):
    return f"A_({self.a},{self.b})"

AffineCipher.__add__ = affine_add
AffineCipher.__mul__ = affine_mul
AffineCipher.__rmul__ = affine_rmul
AffineCipher.__neg__ = affine_neg
AffineCipher.__pow__ = affine_pow
AffineCipher.__str__ = affine_str

test_1 = AffineCipher(5, 3, alphabets["qwerty"]) @ AffineCipher(7, 2, alphabets["qwerty"])
test_2 = AffineCipher(5, 2, alphabets["ambidextrous"]) ** 4
test_3 = 5 * AffineCipher(3, 2)
test_4 = -AffineCipher(3, 2)
test_5 = AffineCipher(4, 2, alphabets["vowels"]) + AffineCipher(4, 1, alphabets["vowels"])
test_6 = AffineCipher(5, 3, alphabets["qwerty"]) @ AffineCipher(8, 2, alphabets["consonants"])

ciphers_b2 = [test_1, test_2, test_3, test_4, test_5, test_6]
run_cipher_tests(ciphers_b2, messages)


Cipher: (A_(5,3) ∘ A_(7,2))
[bond] Original: Agent 007: "Shaken, not stirred!"
[bond] Encoded : Mpycb 007: "Olmayc, cib obnggyk!"
[bond] Decoded : Agent 007: "Shaken, not stirred!"
--------------------------------------------------
[holmes] Original: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
[holmes] Encoded : [Mb 221D Dmayg Obgyyb] Liztyo (bi Vmboic): "Yzytycbmgu, tu kymg Vmboic!"
[holmes] Decoded : [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
--------------------------------------------------
[joker] Original: Why so serious?! Let's put a smile on that face.
[joker] Encoded : Vlu oi oygniho?! Zyb'o jhb m otnzy ic blmb qmxy.
[joker] Decoded : Why so serious?! Let's put a smile on that face.
--------------------------------------------------
[turing] Original: In 1950, Turing asked: "Can machines think?"
[turing] Encoded : Nc 1950, Bhgncp moayk: "Xmc tmxlncyo blnca?"
[turing] Decoded : In 1950, Turing asked: "Can machines thin

**[Justify]**

As in the earlier tasks, the main design choice was to keep the implementation mathematically faithful while reusing the same parent-class infrastructure. The affine-cipher operations are expressed as operator overloads because that mirrors the specification and keeps the code readable: addition, multiplication, negation, and composition all look like the algebra they represent.

For addition I require the same alphabet and the same multiplier $a$, because the specification only defines $A_{a,b_1,\sigma} + A_{a,b_2,\sigma}$ when the multiplier is shared. If either condition fails, the code raises an error rather than inventing an operation that is not mathematically defined. Scalar multiplication and negation are simpler: they only change the offset, so they can be implemented by scaling or negating $b$.

**What restriction on $a$ is required?**

The required restriction is that $a$ must be coprime with the alphabet length $n$:
$$
\gcd(a,n)=1.
$$

That condition is necessary and sufficient for decoding, because only then does $a$ have a modular multiplicative inverse. If $\gcd(a,n) \neq 1$, the affine map is not bijective and more than one plaintext letter can produce the same ciphertext letter, so the original message cannot be recovered uniquely.

**How decoding works**

Encoding is
$$
E(x)=ax+b \pmod n,
$$

so decoding solves for $x$ by applying the inverse of $a$:
$$
D(y)=a^{-1}(y-b) \pmod n.
$$

The offset $b$ is not restricted for invertibility; any value of $b$ is allowed. It only shifts the result. The crucial requirement is the invertibility of the multiplier.

**Concrete examples**

For the coursework example $A_{7,3}$ on the default alphabet, $7^{-1}\equiv 15 \pmod{26}$, so decoding is `x = 15(y-3) mod 26`. For instance, `a` encodes to `d`, and decoding `d` recovers `a`. For $A_{5,8}$, $5^{-1}\equiv 21 \pmod{26}`, so the same logic applies and the cipher can be reversed exactly.

This is why the constructor checks coprimality before accepting the cipher: it prevents invalid choices from being used at all, which is better than allowing a cipher that cannot be decoded correctly.

**Error handling clarity:** `AffineCipher` and its operators raise explicit `ValueError` exceptions for non-coprime multipliers, mismatched alphabets, or incompatible multipliers in addition, and raise `TypeError` when operand types are invalid.

## Task B3: Compositions on Substitution Ciphers &nbsp;&nbsp; [2 marks]

In [9]:
# 1. Define the power method as a standalone function
def substitution_power(self, n):
    """Overloads '**' (S^n) for Substitution Ciphers."""
    if not isinstance(n, int) or n < 0:
        raise ValueError("Power must be a non-negative integer.")
    
    if n == 0:
        # S^0 is the identity cipher
        return SubstitutionCipher(self.alphabet, self.alphabet)
        
    # Calculate the new permutation string mathematically
    new_permuted = self.alphabet
    for _ in range(n):
        new_permuted = self.encode(new_permuted)
        
    return SubstitutionCipher(new_permuted, self.alphabet)

SubstitutionCipher.__pow__ = substitution_power

default_alpha = "abcdefghijklmnopqrstuvwxyz"
default_rev = default_alpha[::-1]
default_split = default_alpha[::2] + default_alpha[1::2][::-1]

# Helper definitions for custom alphabets
qwerty_rev = alphabets["qwerty"][::-1]
vowels_split = alphabets["vowels"][::2] + alphabets["vowels"][1::2][::-1]
ambi_split = alphabets["ambidextrous"][::2] + alphabets["ambidextrous"][1::2][::-1]

# 1. Composition: S_reverse o S_split (Default alphabet)
test_1 = SubstitutionCipher(default_rev) @ SubstitutionCipher(default_split)

# 2. Mismatched Composition: S_{reverse, qwerty} o S_{split, vowels}
test_2 = SubstitutionCipher(qwerty_rev, alphabets["qwerty"]) @ SubstitutionCipher(vowels_split, alphabets["vowels"])

# 3. Power: (S_{split, ambidextrous})^3
test_3 = SubstitutionCipher(ambi_split, alphabets["ambidextrous"]) ** 3

ciphers_b3 = [test_1, test_2, test_3]
run_cipher_tests(ciphers_b3, messages)


Cipher: (SubstitutionCipher(perm='zyxwvutsrqponmlkjihgfedcba', alphabet='abcdefghijklmnopqrstuvwxyz') ∘ SubstitutionCipher(perm='acegikmoqsuwyzxvtrpnljhfdb', alphabet='abcdefghijklmnopqrstuvwxyz'))
[bond] Original: Agent 007: "Shaken, not stirred!"
[bond] Encoded : Znram 007: "Klzfra, acm kmjiirt!"
[bond] Decoded : Agent 007: "Shaken, not stirred!"
--------------------------------------------------
[holmes] Original: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
[holmes] Encoded : [Zm 221X Xzfri Kmirrm] Lcdbrk (mc Szmkca): "Rdrbramziw, bw trzi Szmkca!"
[holmes] Decoded : [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
--------------------------------------------------
[joker] Original: Why so serious?! Let's put a smile on that face.
[joker] Encoded : Slw kc krijcok?! Drm'k eom z kbjdr ca mlzm pzvr.
[joker] Decoded : Why so serious?! Let's put a smile on that face.
--------------------------------------------------
[turing] Original:

**[Justify]**

I used the same style of design as in the earlier tasks: the cipher logic itself is kept small, while the surrounding infrastructure provides the reusable encode/decode pipeline. For substitution ciphers, that means the interesting part is the permutation, not the text traversal, so the class focuses on permutation composition and inverse lookup.

The power operation is implemented by repeatedly applying the substitution to the alphabet and then rebuilding a new `SubstitutionCipher` from the resulting permutation string. That is a good choice because it computes the actual composed permutation directly, rather than building a chain of wrappers whose only purpose would be to re-apply the same mapping at runtime. In other words, the power is collapsed into one concrete cipher object.

**What is $S_{\texttt{reverse}}^2$?**

`S_{\texttt{reverse}}` is its own inverse. Reversing an alphabet once sends the first letter to the last, the second to the second-last, and so on; reversing again returns every character to its original position. Therefore
$$
S_{\texttt{reverse}}^2 = S_{\texttt{identity}},
$$

the identity substitution cipher on the same alphabet.

A concrete example makes this immediate: `a` maps to `z` under the reverse cipher, and applying the reverse cipher again maps `z` back to `a`. The same is true for every other letter, so the second power leaves the alphabet unchanged.

That is also why the implementation of power can be checked by repeated composition: if the result of applying the reverse permutation twice is the original alphabet, then the produced cipher is the identity permutation.

**Error handling clarity:** this task raises clear `ValueError` exceptions for invalid power arguments (for example negative or non-integer exponents), and substitution construction itself raises `ValueError` if a permutation is not a valid bijection.

## Task B4: Mixed Combination &nbsp;&nbsp; [2 marks]

In [10]:
def composite_power(self, n):
    """Overloads '**' so a composed chain can be repeated."""
    if not isinstance(n, int) or n < 1:
        raise ValueError("Power for CompositeCipher must be a positive integer.")
    
    result = self
    for _ in range(n - 1):
        result = result @ self  # Chain it with itself
    return result

# 1. Define the binary subtraction method
def caesar_subtraction(self, other):
    """Overloads the binary '-' operator (C1 - C2)."""
    if not isinstance(other, CaesarCipher):
        raise TypeError("Can only subtract another CaesarCipher.")
    if self.alphabet != other.alphabet:
        raise ValueError("Forbidden: Subtraction requires identical cipher alphabets.")
    
    return CaesarCipher(self.shift - other.shift, self.alphabet)

# 2. Inject it directly into the existing class
CaesarCipher.__sub__ = caesar_subtraction

# Inject the method dynamically
CompositeCipher.__pow__ = composite_power

# 1. Choose Distinct Alphabets
sigma = "abcdefghijklmnopqrstuvwxyz" # default
alpha = alphabets["qwerty"]          # qwerty
beta = alphabets["consonants"]       # consonants (length 21)
gamma = alphabets["vowels"]          # vowels (length 5)

# Pre-compute the S_split permutation for the alpha alphabet
alpha_split = alpha[::2] + alpha[1::2][::-1]

# 2. Build the individual components based on our chosen parameters
# Caesar part: C_{2,sigma} + 4C_{5,sigma} - 2C_{7,sigma}
caesar_part = CaesarCipher(2, sigma) + 4 * CaesarCipher(5, sigma) - 2 * CaesarCipher(7, sigma)

# Substitution part: (S_{split,alpha})^2
sub_part = SubstitutionCipher(alpha_split, alpha) ** 2

# Affine parts: n = 3. 
# A1: a1=4, b1=2 on beta. A2: a2=2, b2=1 on gamma.
affine_1 = 3 * AffineCipher(4, 2, beta)
affine_2 = AffineCipher(2, 1, gamma)

# 3. Assemble the final master cipher using standard algebraic syntax!
# Master: ((Caesar o Sub)^2) o (Affine1 o Affine2)
master_cipher = ((caesar_part @ sub_part) ** 2) @ (affine_1 @ affine_2)

# 4. Run the final test
print(f"Testing Mixed Combination: {master_cipher}\n")
run_cipher_tests([master_cipher], messages)

Testing Mixed Combination: (((C_(8) ∘ SubstitutionCipher(perm='qtodjxnbzhsirwypfkcmvlgaue', alphabet='qwertyuiopasdfghjklzxcvbnm')) ∘ (C_(8) ∘ SubstitutionCipher(perm='qtodjxnbzhsirwypfkcmvlgaue', alphabet='qwertyuiopasdfghjklzxcvbnm'))) ∘ (A_(4,6) ∘ A_(2,1)))


Cipher: (((C_(8) ∘ SubstitutionCipher(perm='qtodjxnbzhsirwypfkcmvlgaue', alphabet='qwertyuiopasdfghjklzxcvbnm')) ∘ (C_(8) ∘ SubstitutionCipher(perm='qtodjxnbzhsirwypfkcmvlgaue', alphabet='qwertyuiopasdfghjklzxcvbnm'))) ∘ (A_(4,6) ∘ A_(2,1)))
[bond] Original: Agent 007: "Shaken, not stirred!"
[bond] Encoded : Brxgw 007: "Vdbkxg, gnw vwahhxy!"
[bond] Decoded : Agent 007: "Shaken, not stirred!"
--------------------------------------------------
[holmes] Original: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
[holmes] Encoded : [Bw 221C Cbkxh Vwhxxw] Dnijxv (wn Pbwvng): "Xixjxgwbhe, je yxbh Pbwvng!"
[holmes] Decoded : [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
---------------

**[Justify]**

This part is mainly about showing that the code handles a genuinely mixed algebraic expression rather than one cipher type in isolation. I therefore chose distinct alphabets and distinct parameters for the component ciphers so that the expression has non-trivial structure but still remains valid under the rules of the coursework.

My choices were: $\sigma$ as the default alphabet, $\alpha$ as `qwerty`, $\beta$ as `consonants`, and $\gamma$ as `vowels`. These are all distinct, as required. I chose $a=2$, $b=5$, and $c=7$ for the Caesar terms so the three shifts are different. I chose $a_1=4$ and $a_2=2$ for the affine parts, again keeping them distinct. I set $n=3$ so that the outer repetition is not trivial and the composition actually has to do real work.

The restrictions come from the component ciphers themselves. For the Caesar part, addition and scalar multiplication only make sense on the same alphabet, so the same `σ` must be used throughout that arithmetic. For the affine part, the multipliers must each be coprime with their respective alphabet lengths so that decoding remains possible. Composition is allowed even when alphabets differ, because the output of one cipher is just fed into the next one as text.

**How the code handles the combination**

The operator overloading lets Python evaluate the expression in the same order as the mathematics: arithmetic collapses the Caesar terms, the substitution power is evaluated as repeated composition, and the affine ciphers are composed sequentially. The `CompositeCipher` wrapper then handles the outer chaining between different cipher families.

This design is useful because it separates mathematical simplification from text processing. For example, the Caesar part is reduced to a single effective shift rather than repeatedly re-encoding messages, which keeps the implementation efficient and makes the output easier to reason about.

The main benefit of this approach is that it demonstrates the algebraic structure required by the task while still preserving correctness for actual message encoding and decoding.

**Error handling clarity:** this mixed expression still uses explicit task-level errors (`TypeError` for invalid operand types, `ValueError` for invalid parameters such as mismatched alphabets or non-coprime affine multipliers), so invalid combinations fail fast with clear diagnostics.

---
# Part C: Affine Hill Cipher &nbsp;&nbsp; [4 marks]

## Task C1: Affine Hill Cipher &nbsp;&nbsp; [2 marks]

In [11]:
class AffineHillCipher:
    """Block cipher implementing the Affine Hill cipher H_{A, b, sigma}."""

    def __init__(self, A, b, alphabet="abcdefghijklmnopqrstuvwxyz"):
        self.A = np.array(A, dtype=int)
        self.b = np.array(b, dtype=int)
        self.n = len(self.b)
        self.alphabet = alphabet
        self.k = len(alphabet)
        self.alphabet_set = set(alphabet)

        if self.A.shape != (self.n, self.n):
            raise ValueError(f"Matrix A must be {self.n}x{self.n}")

        try:
            self.A_inv = np.array(Matrix(self.A).inv_mod(self.k).tolist(), dtype=int)
        except Exception:
            raise ValueError(f"Matrix A is not invertible modulo {self.k}")

    def _collect_cipher_chars(self, text):
        chars = []
        positions = []
        for index, char in enumerate(text):
            lower = char.lower()
            if lower in self.alphabet_set:
                chars.append(lower)
                positions.append((index, char.isupper()))
        return chars, positions

    def _transform_text(self, text, matrix, offset):
        chars, positions = self._collect_cipher_chars(text)
        result = list(text)

        for start in range(0, len(chars), self.n):
            block = chars[start:start + self.n]
            if len(block) < self.n:
                continue  # leave incomplete final block unchanged

            vec = np.array([self.alphabet.index(ch) for ch in block], dtype=int)
            transformed = (matrix @ vec + offset) % self.k

            for idx_in_block, value in enumerate(transformed):
                pos, was_upper = positions[start + idx_in_block]
                new_char = self.alphabet[int(value)]
                result[pos] = new_char.upper() if was_upper else new_char

        return "".join(result)

    def encode(self, text):
        return self._transform_text(text, self.A, self.b)

    def decode(self, text):
        decode_offset = (-self.A_inv @ self.b) % self.k
        return self._transform_text(text, self.A_inv, decode_offset)

    def __str__(self):
        return f"AffineHillCipher(n={self.n}, alphabet='{self.alphabet}')"

    def __repr__(self):
        return self.__str__()


A = [[2, 1], [1, 1]]
b = [1, 0]

hill_c1 = AffineHillCipher(A, b, alphabets["vowels"])

print("=" * 70)
print("TASK C1: Affine Hill Cipher H_{A, b, vowels}")
print("=" * 70)
print(f"Matrix A = {A}")
print(f"Vector b = {b}")
print(f"Alphabet: {alphabets['vowels']}")
print()

for name, msg in messages.items():
    encoded_msg = hill_c1.encode(msg)
    decoded_msg = hill_c1.decode(encoded_msg)
    print(f"[{name}] Original: {msg}")
    print(f"[{name}] Encoded : {encoded_msg}")
    print(f"[{name}] Decoded : {decoded_msg}")
    print("-" * 70)
    assert msg == decoded_msg, f"Decoding failed for {name}!"

print("\nAll C1 tests passed!")

TASK C1: Affine Hill Cipher H_{A, b, vowels}
Matrix A = [[2, 1], [1, 1]]
Vector b = [1, 0]
Alphabet: aeiou

[bond] Original: Agent 007: "Shaken, not stirred!"
[bond] Encoded : Igent 007: "Shiken, nut starred!"
[bond] Decoded : Agent 007: "Shaken, not stirred!"
----------------------------------------------------------------------
[holmes] Original: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
[holmes] Encoded : [Et 221B Bakur Striet] Hulmes (tu Wutson): "Ulimontery, my doer Wutson!"
[holmes] Decoded : [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
----------------------------------------------------------------------
[joker] Original: Why so serious?! Let's put a smile on that face.
[joker] Encoded : Why so suroaas?! Lat's put u smelo in thot fice.
[joker] Decoded : Why so serious?! Let's put a smile on that face.
----------------------------------------------------------------------
[turing] Original: In 1950, Turing asked: "Can ma

**[Justify]**

I designed `AffineHillCipher` around the structure of the block cipher described in the specification: it works on blocks of length $n$, applies the same transformation to every full block, leaves non-alphabet characters untouched, and preserves the case of each letter by position. That makes the implementation faithful to the coursework definition while still being reusable for any square matrix size and any valid alphabet.

The main implementation choice is to separate the block-processing logic from the matrix arithmetic. `_collect_cipher_chars` extracts only the cipher-alphabet characters and records where they came from, while `_transform_text` applies the matrix computation block by block and writes the transformed characters back into the original positions. This is a clean design because it keeps the cipher logic in one place and makes both encoding and decoding use the same pipeline.

**How decoding works**

For a block vector $\mathbf{x}$, the cipher computes
$$
\mathbf{y} = A\mathbf{x} + \mathbf{b} \pmod{k}.
$$

To decode, we solve for $\mathbf{x}$ by subtracting the offset and applying the modular inverse of $A$:
$$
\mathbf{x} = A^{-1}(\mathbf{y} - \mathbf{b}) \pmod{k}.
$$

That is exactly what the code does by precomputing `A_inv` with `Matrix(...).inv_mod(k)` and then using it in `decode`. Precomputing the inverse is a sensible choice because it avoids repeating modular matrix inversion every time a message is decoded.

**Restriction on A**

The matrix must be square of size $n \times n$ and invertible modulo $k$, where $k$ is the alphabet length. Equivalently, the determinant must be coprime with $k$. If this is not true, the block transformation is not bijective and decoding cannot recover the original block uniquely.

**Other constraints**

The offset vector `b` must have length $n$ so that it matches the block size. The alphabet must contain distinct characters, just as in the earlier ciphers. Incomplete final blocks are left unchanged because the specification explicitly says they should not be padded or forced through the matrix transformation.

**How to build another valid example**

A different working example only needs a different square matrix $A$ that is invertible modulo the chosen alphabet length, together with any length-$n$ offset vector `b`. For example, on a five-letter alphabet, any matrix with determinant not divisible by $5$ will work. That gives the user a large family of valid choices rather than one fixed matrix.

For the coursework example, the chosen matrix over the vowels alphabet is valid because it is invertible modulo $5$, so the cipher can be decoded exactly.

**Error handling clarity:** `AffineHillCipher.__init__` raises explicit `ValueError` exceptions when dimensions are inconsistent (for example `A` not being $n\times n$) or when `A` is not invertible modulo the alphabet length, so invalid Hill-cipher configurations are rejected immediately.

## Task C2: Operations on Affine Hill Ciphers &nbsp;&nbsp; [2 marks]

In [12]:
# Extend AffineHillCipher with operator overloading for Task C2

def hill_add(self, other):
    """Add two Affine Hill ciphers with the same matrix A."""
    if not isinstance(other, AffineHillCipher):
        raise TypeError("Can only add another AffineHillCipher.")
    if self.alphabet != other.alphabet:
        raise ValueError("Addition requires identical cipher alphabets.")
    if not np.array_equal(self.A, other.A):
        raise ValueError("Addition requires identical matrix A.")

    new_b = (self.b + other.b) % self.k
    return AffineHillCipher(self.A, new_b, self.alphabet)


def hill_neg(self):
    """Negate an Affine Hill cipher."""
    new_b = (-self.b) % self.k
    return AffineHillCipher(self.A, new_b, self.alphabet)


def hill_pow(self, n):
    """Compose an Affine Hill cipher with itself n times."""
    if not isinstance(n, int) or n < 0:
        raise ValueError("Power must be a non-negative integer.")

    if n == 0:
        identity_A = np.eye(self.n, dtype=int)
        identity_b = np.zeros(self.n, dtype=int)
        return AffineHillCipher(identity_A, identity_b, self.alphabet)

    A_power = np.eye(self.n, dtype=int)
    for _ in range(n):
        A_power = (A_power @ self.A) % self.k

    b_sum = np.zeros(self.n, dtype=int)
    A_temp = np.eye(self.n, dtype=int)
    for _ in range(n):
        b_sum = (b_sum + A_temp @ self.b) % self.k
        A_temp = (A_temp @ self.A) % self.k

    return AffineHillCipher(A_power, b_sum, self.alphabet)


AffineHillCipher.__add__ = hill_add
AffineHillCipher.__neg__ = hill_neg
AffineHillCipher.__pow__ = hill_pow
AffineHillCipher.__matmul__ = lambda self, other: CompositeCipher(self, other)

print("\n" + "=" * 70)
print("TASK C2: Operations on Affine Hill Ciphers")
print("=" * 70)

# Reuse the C1 example as the base cipher, then add a 3x3 example
A1 = [[2, 1], [1, 1]]
b1 = [1, 0]
A2 = [[2, 1, 0], [1, 1, 1], [0, 1, 1]]
b2 = [0, 1, 2]

hill_c1_op = AffineHillCipher(A1, b1, alphabets["vowels"])
hill_c2_op = AffineHillCipher(A2, b2, alphabets["vowels"])

print(f"Base matrix A1 = {A1}")
print(f"Base vector b1 = {b1}")
print(f"Second matrix A2 = {A2}")
print(f"Second vector b2 = {b2}")

test_c2_1 = hill_c1_op @ hill_c2_op
test_c2_2 = hill_c1_op ** 3
test_c2_3 = -hill_c1_op
test_c2_4 = hill_c1_op + AffineHillCipher(A1, [2, 3], alphabets["vowels"])

ciphers_c2 = [
    ("Composition: H_{A,b} ∘ H_{A2,b2}", test_c2_1),
    ("Power: (H_{A,b})^3", test_c2_2),
    ("Negation: -H_{A,b}", test_c2_3),
    ("Addition: H_{A,b} + H_{A,b'}", test_c2_4)
]

for description, cipher in ciphers_c2:
    print(f"\n{description}")
    print("-" * 70)
    for name, msg in messages.items():
        encoded_msg = cipher.encode(msg)
        decoded_msg = cipher.decode(encoded_msg)
        print(f"[{name}] Original: {msg}")
        print(f"[{name}] Encoded : {encoded_msg}")
        print(f"[{name}] Decoded : {decoded_msg}")
        print("-" * 70)
        assert msg == decoded_msg, f"Decoding failed for {name}!"

print("\nAll C2 tests passed!")


TASK C2: Operations on Affine Hill Ciphers
Base matrix A1 = [[2, 1], [1, 1]]
Base vector b1 = [1, 0]
Second matrix A2 = [[2, 1, 0], [1, 1, 1], [0, 1, 1]]
Second vector b2 = [0, 1, 2]

Composition: H_{A,b} ∘ H_{A2,b2}
----------------------------------------------------------------------
[bond] Original: Agent 007: "Shaken, not stirred!"
[bond] Encoded : Agont 007: "Shikon, nit sturred!"
[bond] Decoded : Agent 007: "Shaken, not stirred!"
----------------------------------------------------------------------
[holmes] Original: [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
[holmes] Encoded : [Ot 221B Bikar Streut] Hilmes (ta Wotsin): "Ilamontary, my dior Wutsun!"
[holmes] Decoded : [At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"
----------------------------------------------------------------------
[joker] Original: Why so serious?! Let's put a smile on that face.
[joker] Encoded : Why si sureaes?! Let's put a smulu en that fice.
[joker

**[Justify]**

I kept the same design principles as in the rest of the notebook, but here they matter at the block-cipher level. Composition is handled by the `CompositeCipher` wrapper, power is handled by repeated composition, negation flips the offset vector, and addition combines ciphers with the same matrix `A` by adding their offset vectors. That mirrors the algebra described in the specification and keeps the implementation close to the mathematics.

**How I chose $A_2$**

I chose
$$
A_2 = \begin{pmatrix} 2 & 1 & 0 \\ 1 & 1 & 1 \\ 0 & 1 & 1 \end{pmatrix}, \qquad \mathbf{b}_2 = \begin{pmatrix} 0 \\ 1 \\ 2 \end{pmatrix}
$$

because it is a genuinely different working example from the C1 matrix while still being valid modulo $5$. Its determinant is $-1$, which is congruent to $4 \pmod 5$, so it is invertible modulo the vowels alphabet length. That means it produces a reversible cipher, but with a different diffusion pattern from the 2x2 example in C1.

The point of choosing a 3x3 matrix is that it shows the code is dimension-aware. The implementation cannot be hard-coded for a 2x2 case, because it must correctly handle different block sizes and still preserve reversibility. Using a larger block also gives stronger mixing between characters, which makes the demonstration less trivial.

**How the code handles the combination**

The composed expressions work because each affine Hill cipher remains individually invertible, and the wrapper applies them in sequence to the encoded text. For power, the repeated composition is collapsed into a new affine Hill cipher with matrix power $A^n$ and the corresponding accumulated offset term. Negation and addition operate only on the offset vector, provided the underlying matrix and alphabet match.

The key restriction throughout is that every component cipher must remain valid on the chosen alphabet. In particular, the matrix must stay invertible modulo the alphabet size, and the block size used by the matrix must match the length of the offset vector. Once those conditions are met, the code can combine the ciphers safely and still decode the result exactly.

**Error handling clarity:** these operations raise explicit exceptions to make invalid usage obvious: `TypeError` when the other operand is not an `AffineHillCipher`, and `ValueError` when alphabet or matrix-compatibility constraints are violated (for example adding ciphers with different `A` or incompatible alphabets).